In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder

In [ ]:
df = pd.read_csv("../data/train.csv")
df.head()

In [ ]:
df.describe()
print(df.info())

In [ ]:
df.drop_duplicates(inplace=True)
df.isnull().sum().sort_values(ascending=False)

In [ ]:
# ==========================================================
# Handling Missing Values
# ==========================================================

# Fill categorical features where NaN means the feature
# does not exist rather than missing information.

fill_values = {
    "PoolQC": "NoPool",
    "MiscFeature": "NoMiscFeature",
    "Alley": "NoAlley",
    "Fence": "NoFence",
    "MasVnrType": "NoMasonry",
    "FireplaceQu": "NoFireplace",
    "GarageType": "NoGarage",
    "GarageFinish": "NoGarage",
    "GarageQual": "NoGarage",
    "GarageCond": "NoGarage",
    "BsmtQual": "NoBasement",
    "BsmtCond": "NoBasement",
    "BsmtExposure": "NoBasement",
    "BsmtFinType1": "NoBasement",
    "BsmtFinType2": "NoBasement"
}

for col, value in fill_values.items():
    df[col] = df[col].fillna(value)

# ==========================================================
# Maintain Garage Consistency
# ==========================================================

garage_cat = [
    "GarageType",
    "GarageFinish",
    "GarageQual",
    "GarageCond"
]

for col in garage_cat:
    df.loc[df["GarageType"] == "NoGarage", col] = "NoGarage"

# ==========================================================
# LotFrontage Imputation
# ==========================================================

df["LotFrontage"] = (
    df.groupby("Neighborhood")["LotFrontage"]
    .transform(lambda x: x.fillna(x.median()))
)

# In case an entire neighborhood has missing values
df["LotFrontage"] = df["LotFrontage"].fillna(df["LotFrontage"].median())

# ==========================================================
# Masonry Veneer Area
# ==========================================================

df.loc[df["MasVnrType"] == "NoMasonry", "MasVnrArea"] = 0
df["MasVnrArea"] = df["MasVnrArea"].fillna(0)

# ==========================================================
# Electrical System
# ==========================================================

df["Electrical"] = (
    df.groupby("Neighborhood")["Electrical"]
    .transform(lambda x: x.fillna(x.mode()[0]))
)

# Fallback if any NaNs remain
df["Electrical"] = df["Electrical"].fillna(df["Electrical"].mode()[0])

# ==========================================================
# Maintain Garage Consistency (Numerical)
# ==========================================================

garage_num = [
    "GarageYrBlt",
    "GarageCars",
    "GarageArea"
]

for col in garage_num:
    df.loc[df["GarageType"] == "NoGarage", col] = 0

# ==========================================================
# Final Check
# ==========================================================

print(df.isna().sum()[df.isna().sum() > 0])

In [ ]:
# =========================
# Feature Engineering
# =========================

# Total house area
df["TotalSF"] = (
        df["TotalBsmtSF"] +
        df["1stFlrSF"] +
        df["2ndFlrSF"]
)

# House age
df["HouseAge"] = df["YrSold"] - df["YearBuilt"]

# Years since last remodel
df["RemodelAge"] = df["YrSold"] - df["YearRemodAdd"]

# Garage age
df["GarageAge"] = np.where(
    df["GarageYrBlt"] == 0,
    0,
    df["YrSold"] - df["GarageYrBlt"]
)

# Total bathrooms
df["TotalBathrooms"] = (
        df["FullBath"] +
        0.5 * df["HalfBath"] +
        df["BsmtFullBath"] +
        0.5 * df["BsmtHalfBath"]
)

# Total porch area
df["TotalPorchSF"] = (
        df["OpenPorchSF"] +
        df["EnclosedPorch"] +
        df["3SsnPorch"] +
        df["ScreenPorch"]
)

# Total outdoor area
df["TotalOutdoorSF"] = (
        df["WoodDeckSF"] +
        df["TotalPorchSF"]
)

# Binary features
df["HasGarage"] = (df["GarageArea"] > 0).astype(int)
df["HasBasement"] = (df["TotalBsmtSF"] > 0).astype(int)
df["HasFireplace"] = (df["Fireplaces"] > 0).astype(int)
df["HasPool"] = (df["PoolArea"] > 0).astype(int)
df["Has2ndFloor"] = (df["2ndFlrSF"] > 0).astype(int)
df["HasRemodel"] = (df["YearBuilt"] != df["YearRemodAdd"]).astype(int)

# Optional
df["LotAreaPerRoom"] = df["LotArea"] / df["TotRmsAbvGrd"]

In [ ]:
# TO Check if all the columns are cleaned and NAN is filled
df.isnull().sum()[df.isnull().sum() > 0]

In [ ]:
df.to_csv("../data/Cleaned_Housing_Dataset.csv" , sep = ',')